<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%205/Aula_5_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 5.2 — Monitorando e gerenciando agentes

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 5 — Gerenciando agentes com AgentOS

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 5.1**:
- agentOS
- banco SQLite
- único agente plugado

Esta aula resolve algumas coisas:

1. **Plugar time e workflow** ao AgentOS
2. **Memória de longo prazo**
3. **Observabilidade via Python**  



## Setup


In [17]:
!pip install -U "agno[os]" openai tavily-python wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=d68f4e6dc947cd79ddd39c2173960162ab09c58bee95f273120e2840fdb1a8a0
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [18]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — A dor sem memória persistente

Vamos partir de um Treinador básico **com banco, mas sem memória persistente ativada**. Vamos demonstrar o problema antes de resolver.


In [19]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.sqlite import SqliteDb
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

# Banco compartilhado (mesmo padrão da 5.1)
db = SqliteDb(db_file="/tmp/scoutai_5_2.db")

modelo_treinador = OpenAIChat(id="gpt-5.4")

treinador_sem_memoria = Agent(
    name="Treinador",
    role="Voz oficial do ScoutAI FC",
    model= modelo_treinador,
    db=db,
    tools=[TavilyTools()],
    instructions=[
        "Você é o Treinador do ScoutAI FC.",
        "Responda sobre futebol em português do Brasil, com tom profissional.",
    ],
    add_history_to_context=True,    # mantém contexto da sessão atual
    num_history_runs=3,
    # Note: SEM enable_user_memories ainda
    markdown=True,
)

### Sessão 1 — usuário se apresenta

Imagine um torcedor abrindo o ScoutAI FC pela primeira vez. Ele se apresenta e dá pistas sobre seu perfil de torcedor.


In [20]:
treinador_sem_memoria.print_response(
    "Oi! Sou o Fabio, torço pro Botafogo desde criança. "
    "Adoro times com pontas rápidos e jogo vertical, sem muita posse de bola.",
    user_id="fabio-torcedor",
    session_id="primeira-conversa",
    stream=True,
)

Output()

### Sessão 2 — em outro momento, outra pergunta

Agora simule que o Fabio voltou ao app **dias depois**. Outra sessão, outro `session_id`. Mesma pessoa, contexto diferente.


In [21]:
treinador_sem_memoria.print_response(
    "Que esquema tático você recomendaria pra mim?",
    user_id="fabio-torcedor",
    session_id="segunda-conversa",   # ← session_id diferente
    stream=True,
)

Output()

### O que acabou de acontecer



`add_history_to_context=True` mantém o histórico **dentro de uma sessão**. Mas entre sessões diferentes, **tudo se perde**.



---

## Passo 2 — Ativando memória de longo prazo

Uma linha resolve: `enable_user_memories=True`. Com isso, o Agno automaticamente:

1. **Extrai fatos**
2. **Guarda no banco**
3. **Recupera**


In [22]:
treinador_com_memoria = Agent(
    name="Treinador",
    role="Voz oficial do ScoutAI FC",
    model=OpenAIChat(id="gpt-4o-mini"),
    db=db,
    tools=[TavilyTools()],
    instructions=[
        "Você é o Treinador do ScoutAI FC.",
        "Responda sobre futebol em português do Brasil, com tom profissional.",
        "Quando souber preferências do usuário (time do coração, estilo de jogo favorito), "
        "considere essas preferências ao recomendar.",
    ],
    add_history_to_context=True,
    num_history_runs=3,
    enable_user_memories=True,       # ← peça nova: extração e persistência de fatos
    markdown=True,
)

### Repetindo a sequência: apresentação + pergunta em outra sessão

Vamos repetir o experimento, agora com memória ativa. Mesmo `user_id`, sessões distintas.


In [23]:
# Sessão 1 — usuário se apresenta
treinador_com_memoria.print_response(
    "Oi! Sou o Fabio, torço pro Botafogo desde criança. "
    "Adoro times com pontas rápidos e jogo vertical, sem muita posse de bola.",
    user_id="fabio-torcedor",
    session_id="conversa-com-memoria-1",
    stream=True,
)

Output()

In [24]:
# Sessão 2 — outra hora, outra pergunta
treinador_com_memoria.print_response(
    "Que esquema tático você recomendaria pra mim?",
    user_id="fabio-torcedor",
    session_id="conversa-com-memoria-2",   # session_id diferente
    stream=True,
)

Output()

---

## Passo 3 — Inspecionando as memórias guardadas

Memória de longo prazo só faz sentido se você **vê** o que está sendo guardado — pra debugar, auditar, ou simplesmente entender o produto. O Agno expõe isso via API:


In [25]:
# Recuperar todas as memórias do Fabio
memorias = treinador_com_memoria.get_user_memories(user_id="fabio-torcedor")

print(f"Total de memórias guardadas: {len(memorias)}\n")
for i, mem in enumerate(memorias, 1):
    print(f"{i}. {mem.memory}")
    if hasattr(mem, "topics") and mem.topics:
        print(f"   Tópicos: {mem.topics}")
    print()

Total de memórias guardadas: 3

1. User's name is Fabio.
   Tópicos: ['name']

2. Fabio has been a fan of Botafogo since childhood.
   Tópicos: ['interests', 'sports']

3. Fabio enjoys teams with fast wingers and vertical play, without much ball possession.
   Tópicos: ['preferences', 'sports']



### O que o Agno está guardando

Note que o Agno **não guarda a frase original** do usuário, guarda **fatos extraídos**.

| Conversa literal (chat history) | Fatos extraídos (memória de longo prazo) |
|---|---|
| *"Sou o Fabio, torço pro Botafogo desde criança..."* | Preferência por Botafogo |
| | Preferência por jogo vertical |
| | Preferência por pontas rápidos |
| | Aversão à posse de bola excessiva |

Por que isso importa:

- **Eficiência**
- **Privacidade**
- **Reuso**


---

## Passo 4 — Plugando time e workflow ao AgentOS



In [27]:
# Time enxuto — Treinador-líder + Olheiro

modelo_olheiro = OpenAIChat(id="gpt-5.4-mini")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas",
    model=modelo_olheiro,
    db=db,
    tools=[TavilyTools(), WikipediaTools()],
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Busca dados verificáveis em fontes externas.",
        "Use Tavily para eventos recentes (últimos jogos, convocações, forma de jogadores).",
        "Use Wikipedia para fatos históricos consolidados.",
        "Retorne dados objetivos — números, datas, nomes, fatos. Não interprete nem opine.",
        "Se a busca falhar, diga claramente em vez de chutar.",
    ],
    markdown=True,
)

from agno.team import Team

modelo_treinador = OpenAIChat(id="gpt-5.4")

time_scoutai = Team(
    name="ScoutAI Time",
    mode='coordinate',
    members=[olheiro],
    model=modelo_treinador,
    db=db,
    instructions=[
        "Você é o Treinador do ScoutAI FC, líder da comissão técnica.",
        "Coordena o Olheiro (busca dados) e o Analista (interpreta dados e gera gráficos).",
        "Quando o usuário pedir dados ou fatos verificáveis, delegue ao Olheiro.",
        "Quando o usuário pedir comparação visual, gráfico ou análise de evolução, "
        "primeiro garanta que o Olheiro trouxe os dados, depois delegue ao Analista para visualizar.",
        "Para perguntas conceituais simples, responda direto sem delegar.",
        "Sempre integre dados e análise antes de responder ao usuário, em português do Brasil.",
    ],
    add_history_to_context=True,
    num_history_runs=3,
    markdown=True,
)

In [28]:
# Workflow mínimo — Olheiro + Treinador
from agno.workflow import Step, Workflow

workflow_escalacao = Workflow(
    name="Recomendação rápida de escalação",
    steps=[
        Step(name="Coleta", agent=olheiro),
        Step(name="Síntese", agent=treinador_com_memoria),
    ],
)

### Criando o AgentOS com os três tipos


In [29]:
from agno.os import AgentOS

agent_os = AgentOS(
    agents=[treinador_com_memoria, olheiro],
    teams=[time_scoutai],
    workflows=[workflow_escalacao],
)

app = agent_os.get_app()
print(f"AgentOS gerado com FastAPI app: {type(app).__name__}")

AgentOS gerado com FastAPI app: FastAPI


### Inspecionando os endpoints gerados

Vamos ver as rotas que apareceram. Note que agora o app tem endpoints **para os três tipos**: agentes, times, workflows.


In [30]:
rotas = sorted(set(route.path for route in app.routes if hasattr(route, "path")))

# Filtrar só rotas que envolvem agente/time/workflow (sem health, docs, etc)
rotas_relevantes = [r for r in rotas if any(t in r for t in ["agent", "team", "workflow"])]

print(f"Total de rotas relevantes: {len(rotas_relevantes)}\n")
for rota in rotas_relevantes[:20]:
    print(f"  {rota}")

Total de rotas relevantes: 22

  /agents
  /agents/{agent_id}
  /agents/{agent_id}/runs
  /agents/{agent_id}/runs/{run_id}
  /agents/{agent_id}/runs/{run_id}/cancel
  /agents/{agent_id}/runs/{run_id}/continue
  /agents/{agent_id}/runs/{run_id}/resume
  /teams
  /teams/{team_id}
  /teams/{team_id}/runs
  /teams/{team_id}/runs/{run_id}
  /teams/{team_id}/runs/{run_id}/cancel
  /teams/{team_id}/runs/{run_id}/continue
  /teams/{team_id}/runs/{run_id}/resume
  /workflows
  /workflows/ws
  /workflows/{workflow_id}
  /workflows/{workflow_id}/runs
  /workflows/{workflow_id}/runs/{run_id}
  /workflows/{workflow_id}/runs/{run_id}/cancel


**O que isso entrega em produção:** o frontend mobile do ScoutAI FC pode chamar:

- `POST /agents/treinador/runs` — chat conversacional
- `POST /teams/scoutai-time/runs` — pergunta delegada ao time
- `POST /workflows/recomendacao-rapida-de-escalacao/runs` — pipeline determinístico

**Tudo via HTTP**, com o mesmo backend cuidando de sessões, memória, persistência. Você não precisou escrever nenhuma rota — o AgentOS gerou todas a partir dos seus objetos Python.


---

## Passo 5 — Observabilidade via Python



In [31]:
# Recuperar todas as sessões do Fabio
sessoes = db.get_sessions(user_id="fabio-torcedor")

print(f"Total de sessões do Fabio: {len(sessoes)}\n")
for i, sessao in enumerate(sessoes, 1):
    print(f"--- Sessão {i} ---")
    print(f"  session_id: {sessao.session_id}")
    print(f"  user_id: {sessao.user_id}")
    if hasattr(sessao, "agent_id") and sessao.agent_id:
        print(f"  agent_id: {sessao.agent_id}")
    if hasattr(sessao, "created_at") and sessao.created_at:
        print(f"  criada em: {sessao.created_at}")
    print()

Total de sessões do Fabio: 5

--- Sessão 1 ---
  session_id: primeira-conversa
  user_id: fabio-torcedor
  agent_id: treinador
  criada em: 1777949234

--- Sessão 2 ---
  session_id: segunda-conversa
  user_id: fabio-torcedor
  agent_id: treinador
  criada em: 1777949247

--- Sessão 3 ---
  session_id: conversa-com-memoria-1
  user_id: fabio-torcedor
  agent_id: treinador
  criada em: 1777949263

--- Sessão 4 ---
  session_id: conversa-com-memoria-2
  user_id: fabio-torcedor
  agent_id: treinador
  criada em: 1777949267

--- Sessão 5 ---
  session_id: debug-demo
  user_id: fabio-torcedor
  agent_id: treinador
  criada em: 1777949272



### Olhando dentro de uma sessão específica

Cada sessão guarda os turnos completos — perguntas, respostas, tool calls, métricas. Vamos ver os primeiros campos da sessão mais recente:


In [32]:
# Pegar a sessão mais recente
if sessoes:
    sessao_recente = sessoes[0]
    print(f"Inspecionando sessão: {sessao_recente.session_id}\n")

    # Dump dos campos disponíveis na sessão
    campos = [c for c in dir(sessao_recente) if not c.startswith("_") and not callable(getattr(sessao_recente, c))]
    print(f"Campos disponíveis ({len(campos)}):")
    for c in campos[:15]:
        valor = getattr(sessao_recente, c)
        valor_str = str(valor)[:80] + "..." if len(str(valor)) > 80 else str(valor)
        print(f"  {c}: {valor_str}")
else:
    print("Nenhuma sessão encontrada — rode os passos anteriores primeiro.")

Inspecionando sessão: primeira-conversa

Campos disponíveis (12):
  agent_data: {'name': 'Treinador', 'agent_id': 'treinador', 'model': {'name': 'OpenAIChat', '...
  agent_id: treinador
  created_at: 1777949234
  metadata: None
  runs: [RunOutput(run_id='bb16abb3-e6f7-465a-9ee1-6e6fc49a6b9e', agent_id='treinador', ...
  session_data: {'session_state': {}, 'session_metrics': {'input_tokens': 1029, 'output_tokens':...
  session_id: primeira-conversa
  summary: None
  team_id: None
  updated_at: 1777951267
  user_id: fabio-torcedor
  workflow_id: None


### O que isso destrava



- **Dashboards customizados**
- **Alertas**
- **Exportação**
- **Análise de qualidade**
- **Compliance**




---

### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Persistência via SqliteDb (Aula 5.1)
├── ✅ Time + Workflow plugados ao AgentOS (esta aula)
├── ✅ Memória de longo prazo por usuário (esta aula)
├── ✅ Observabilidade programática (esta aula)
│
├── ❌ Sem versionamento de prompts                  → Aula 5.3
├── ❌ Sem estratégia de escala explícita            → Aula 5.3
└── ❌ Sem proteção de PII (LGPD)                    → Aula 5.3
```

### Próxima aula

**Aula 5.3 — Governança, escala e boas práticas**

